# Notebook 最小检索闭环：真实端到端体验

本 Notebook 使用项目的正式 HTTP 接口完成：

`课程目录递归发现 → 只读解析预览 → 结构化切块预览 → Qwen Embedding → Qdrant 入库 → 原始 query dense Top K 召回 → 文件/章节/cell 来源回溯`

课程 Notebook 中的代码**不会被执行**。当前正式索引只写入文本表示；视觉表示已经可以在 `chunk_embedding_experiment.ipynb` 中生成，但尚未接入 Qdrant 检索闭环。

运行前需要：

1. 根目录 `.env` 已填写 DashScope API。
2. Qdrant Docker 容器已启动，默认地址为 `http://localhost:6333`。
3. 按顺序执行全部 cell。


In [ ]:
from collections import Counter
from pathlib import Path
import json
import os
import sys

import httpx
from qdrant_client import models

# 自动定位项目根目录，使 Notebook 从项目根目录或 notebooks/ 启动都可运行。
start_dir = Path.cwd().resolve()
PROJECT_ROOT = next(
    (path for path in (start_dir, *start_dir.parents) if (path / "pyproject.toml").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("无法定位 SRAG 项目根目录")
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

from app.config import get_settings
from app.main import create_app
from app.rag.chunker import NotebookChunker
from app.rag.parsers.notebook import NotebookParser
from app.stores.qdrant import QdrantStore

# ===== 用户配置区 =====
COURSE_ID = "CS5489"
COURSE_DIR = get_settings().course_root / "your-course"
QUERY = "贝叶斯公式是什么？"
TOP_K = 5

# None 表示索引目录下所有符合条件的 Notebook；首次体验建议设为 1。
MAX_FILES: int | None = 1
FILE_GLOB = "**/*.ipynb"

# 默认排除 Jupyter 临时目录和历史资料；如需纳入历史资料，可从集合中删除对应名称。
EXCLUDE_PATH_PARTS = {".ipynb_checkpoints", "PreviousDocument"}
PREVIEW_CHUNKS = 8

settings = get_settings()
app = create_app()
parser = NotebookParser.from_settings(settings)
chunker = NotebookChunker.from_settings(settings)

print(f"项目根目录: {PROJECT_ROOT}")
print(f"课程目录: {COURSE_DIR}")
print(f"课程 ID: {COURSE_ID}")
print(f"默认 query: {QUERY}")
print(f"Top K: {TOP_K}")
print(f"DashScope API: {'已配置' if settings.dashscope_api_key else '未配置'}")
print(f"Qdrant: {settings.qdrant_url}/{settings.qdrant_collection}")


## 1. 环境与 Qdrant 检查

这一步不调用 Embedding API，只确认课程目录、凭据和向量库是否可用。


In [ ]:
course_dir = COURSE_DIR.expanduser().resolve()
if not course_dir.is_dir():
    raise FileNotFoundError(f"课程目录不存在: {course_dir}")
if not settings.dashscope_api_key:
    raise RuntimeError(".env 中尚未配置 DASHSCOPE_API_KEY")

store = QdrantStore(settings)
try:
    collection_info = await store.ensure_collection()
finally:
    await store.close()

print("环境检查通过")
print(f"Qdrant collection 状态: {collection_info.status}")
print(f"当前 points: {collection_info.points_count}")
print(f"当前 indexed vectors: {collection_info.indexed_vectors_count}")


## 2. 递归发现课程 Notebook

`MAX_FILES=None` 时，后续索引本列表中的全部文件。API 当前接收的是文件路径列表，目录递归由这个 Notebook 完成。


In [ ]:
all_notebooks = [
    path.resolve()
    for path in sorted(course_dir.glob(FILE_GLOB))
    if path.is_file() and not EXCLUDE_PATH_PARTS.intersection(path.parts)
]
if not all_notebooks:
    raise FileNotFoundError(f"目录中没有找到可用的 .ipynb: {course_dir}")

selected_notebooks = (
    all_notebooks if MAX_FILES is None else all_notebooks[:MAX_FILES]
)
if not selected_notebooks:
    raise ValueError("MAX_FILES 必须为正整数或 None")

print(f"共发现 {len(all_notebooks)} 个可用 Notebook")
print(f"本次选择 {len(selected_notebooks)} 个 Notebook")
for index, path in enumerate(selected_notebooks, 1):
    print(f"[{index:02d}] {path.relative_to(course_dir)}")

if MAX_FILES is not None and len(all_notebooks) > len(selected_notebooks):
    print(f"\n为控制 API 用量，本次未处理其余 {len(all_notebooks) - len(selected_notebooks)} 个文件。")
    print("确认小规模流程后，将 MAX_FILES 改为 None 并从本 cell 重新执行。")


## 3. 解析与切块预览

这里只进行本地只读解析和切块，不调用模型 API。每个文件会输出内容类型、Cell Group、Section 和 Chunk 数量。


In [ ]:
document_ids = []
preview_summaries = []
first_structure = None

for path in selected_notebooks:
    document = parser.parse(path)
    structure = chunker.chunk(document)
    document_ids.append(document.document_id)
    preview_summaries.append(
        {
            "file": str(path.relative_to(course_dir)),
            "document_id": document.document_id,
            "title": document.title,
            "blocks": len(document.blocks),
            "block_types": dict(Counter(block.kind.value for block in document.blocks)),
            "assets": sum(len(block.assets) for block in document.blocks),
            "cell_groups": len(structure.groups),
            "sections": len(structure.sections),
            "chunks": len(structure.chunks),
            "chunk_tokens": {
                "min": min((chunk.token_count for chunk in structure.chunks), default=0),
                "max": max((chunk.token_count for chunk in structure.chunks), default=0),
            },
        }
    )
    if first_structure is None:
        first_structure = structure

print(json.dumps(preview_summaries, ensure_ascii=False, indent=2))

print("\n第一个文件的 Chunk 预览:")
for index, chunk in enumerate(first_structure.chunks[:PREVIEW_CHUNKS], 1):
    preview = " ".join(chunk.embedding_text.split())[:180]
    print(
        f"[{index:02d}] kind={chunk.kind.value} tokens={chunk.token_count} "
        f"section={' > '.join(chunk.section_path) or '(根章节)'} "
        f"cells={chunk.cell_ids}\n     {preview}"
    )


## 4. 通过正式 HTTP 接口完成 Embedding 与 Qdrant 入库

这一 cell 会产生 DashScope API 调用。索引器会为每个文件生成一条 `document_outline` 和多条 `text_content` 表示，并幂等写入 Qdrant。


In [ ]:
async def post_api(path: str, payload: dict) -> dict:
    transport = httpx.ASGITransport(app=app)
    async with httpx.AsyncClient(
        transport=transport,
        base_url="http://srag.local",
        timeout=None,
        
    ) as client:
        response = await client.post(path, json=payload)
        print(f"POST {path} -> HTTP {response.status_code}")
        response.raise_for_status()
        return response.json()

index_result = await post_api(
    "/api/rag/index",
    {
        "course_id": COURSE_ID,
        "paths": [str(path) for path in selected_notebooks],
    },
)
print(json.dumps(index_result, ensure_ascii=False, indent=2))

if index_result["failed_files"]:
    print("\n存在索引失败文件，请先查看 errors；成功文件仍已写入 Qdrant。")


## 5. 查看 Qdrant 中的实际写入结果

按课程和本次文档分别统计 point 数量。重复执行索引时，稳定 ID 会覆盖已有 point，数量通常保持不变。


In [ ]:
store = QdrantStore(settings)
try:
    course_filter = models.Filter(
        must=[
            models.FieldCondition(
                key="course_id",
                match=models.MatchValue(value=COURSE_ID),
            )
        ]
    )
    course_count = await store.client.count(
        collection_name=settings.qdrant_collection,
        count_filter=course_filter,
        exact=True,
    )
    print(f"课程 {COURSE_ID} 当前共有 {course_count.count} 个 points")

    for summary in preview_summaries:
        document_filter = models.Filter(
            must=[
                models.FieldCondition(
                    key="document_id",
                    match=models.MatchValue(value=summary["document_id"]),
                )
            ]
        )
        document_count = await store.client.count(
            collection_name=settings.qdrant_collection,
            count_filter=document_filter,
            exact=True,
        )
        print(
            f"{summary['file']} | document_id={summary['document_id']} "
            f"| points={document_count.count}"
        )
finally:
    await store.close()


## 6. 自由输入 query，跨本次课程文件检索

修改配置 cell 中的 `QUERY`，或者直接修改下面 `await search_course(...)` 的参数。当前使用原始 query 的 Qwen dense embedding 和 Cosine Top K 召回。


In [ ]:
async def search_course(
    query: str,
    *,
    top_k: int = TOP_K,
    restrict_to_selected_files: bool = False,
) -> dict:
    scope = {"course_ids": [COURSE_ID], "document_ids": []}
    if restrict_to_selected_files:
        scope["document_ids"] = document_ids

    result = await post_api(
        "/api/rag/search",
        {"query": query, "scope": scope, "top_k": top_k},
    )
    print(f"query: {result['query']}")
    print(f"召回数量: {len(result['hits'])}\n")

    for rank, hit in enumerate(result["hits"], 1):
        source = hit["source"]
        section = " > ".join(source["section_path"]) or "(根章节)"
        content = " ".join(hit["content"].split())[:500]
        print(f"===== Top {rank} | score={hit['score']:.6f} =====")
        print(f"文件: {source['file']}")
        print(f"document_id: {source['document_id']}")
        print(f"章节: {section}")
        
        print(f"cells: {source['cell_ids']}")
        print(f"chunk_id: {hit['chunk_id']}")
        print(f"内容: {content}\n")
    return result

search_result = await search_course(QUERY, top_k=TOP_K)


## 7. 可选：只在本次选择的文件内检索

当同一 `course_id` 已经索引过其他 Notebook 时，课程范围检索会覆盖它们。将下面 cell 取消注释，可以用 `document_ids` 把范围限定为本次选择的文件。


In [ ]:
# selected_only_result = await search_course(
#     "把这里改成你的问题",
#     top_k=TOP_K,
#     restrict_to_selected_files=True,
# )


## 如何判断结果是否合理

逐条检查：

- **文件范围**：命中是否来自期望课程或指定文件。
- **章节定位**：`section_path` 是否与问题主题一致。
- **Cell 来源**：`cell_ids` 是否能回到原 Notebook 找到证据。
- **内容相关性**：Top 1–Top 5 是否包含回答问题所需的信息。
- **分数用途**：Cosine score 只用于同一检索配置下排序，不是“答案正确率”。

当前限制：

- 正式闭环仅检索已经索引的 Notebook，不会在每次 query 时自动扫描课程目录。
- 目录新增文件后，需要重新运行“文件发现 + 索引”步骤。
- 版本、发布日期可见性、旧版本清理和自动目录同步尚未完成。
- 当前召回为 text dense baseline；sparse/RRF、rerank 和视觉召回仍在后续 checklist 中。
